In [6]:
#Celda 1 — Imports y configuración:

import requests
import pandas as pd
from dotenv import load_dotenv
from datetime import datetime, timedelta

load_dotenv()

BCRP_BASE = "https://estadisticas.bcrp.gob.pe/estadisticas/series/api"

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


In [7]:
# Celda 2 — Función para fechas:

def get_fecha_rango(dias=30): # por defecto la variable días será igual a 30
    hoy = datetime.today() #por ejemplo: 2026-04-16 08:23:45.123456 - Formato: año-mes-día hora:minutos:segundos.microsegundos
    inicio = hoy - timedelta(days=dias) # Hoy (2026-04-16 08:23:45.123456) - Timedelta (30 days, 0:00:00)
    return inicio.strftime("%Y-%m-%d"), hoy.strftime("%Y-%m-%d") #Se le está dando formato de año (4 digitos), mes (2 num), día (2 num).
    #devuelve 2 valores separados por coma, por lo tanto se guarda en una tupla.

fecha_inicio, fecha_fin = get_fecha_rango() # desempaqueta la tupla resultante de ejecutar la función y le asigna 2 variables
print(f"Rango: {fecha_inicio} → {fecha_fin}")

Rango: 2026-03-18 → 2026-04-17


In [8]:
# Celda 3 — Convertir fechas + traer datos del BCRP


MESES_ES = {
    "Ene": "Jan", "Feb": "Feb", "Mar": "Mar", "Abr": "Apr",
    "May": "May", "Jun": "Jun", "Jul": "Jul", "Ago": "Aug",
    "Sep": "Sep", "Oct": "Oct", "Nov": "Nov", "Dic": "Dec"
}

def convertir_fecha_bcrp(fecha_str):
    try:
        partes = fecha_str.split(".")
        if len(partes) == 3:
            dia = partes[0]
            mes = MESES_ES.get(partes[1], partes[1])
            anio = "20" + partes[2]
            return pd.to_datetime(f"{dia} {mes} {anio}", format="%d %b %Y")
        elif len(partes) == 2:
            mes = MESES_ES.get(partes[0], partes[0])
            anio = partes[1]
            return pd.to_datetime(f"01 {mes} {anio}", format="%d %b %Y")
    except Exception:
        return None

def fetch_serie(codigo, fecha_inicio, fecha_fin):
    url = f"{BCRP_BASE}/{codigo}/json/{fecha_inicio}/{fecha_fin}/esp"
    try:
        response = requests.get(url, timeout=10) #se usa el método get para solicitar a la web la información definida en URL
        response.raise_for_status() # Lanza una excepción cuando el status es 400 o superior
        data = response.json() # Transforma response (inicialmente como un file con archivos) a un diccionario entendible por python
        periodos = data.get("periods", []) # equivalente a data["periods"], pero util para enlaces desconocidos, si no encuentra coloca []
        registros = [] 
        for p in periodos:
            fecha_raw = p.get("name")
            valor = p.get("values", [None])[0]
            if valor is not None:
                try:
                    fecha = convertir_fecha_bcrp(fecha_raw)
                    if fecha is not None:
                        registros.append({
                            "fecha": fecha,
                            "valor": float(valor)
                        })
                except (ValueError, TypeError):
                    pass
        df = pd.DataFrame(registros)
        if not df.empty:
            df = df.sort_values("fecha").reset_index(drop=True)
        return df
    except Exception as e:
        print(f"Error: {e}")
        return pd.DataFrame()

print("Funciones listas")

Funciones listas


In [9]:
# Celda 4 — Traer todas las series

def fetch_todas_las_series(series_dict, fecha_inicio, fecha_fin):
    resultados = {}
    for nombre, codigo in series_dict.items():
        df = fetch_serie(codigo, fecha_inicio, fecha_fin)
        resultados[nombre] = df
        if not df.empty:
            ultimo_valor = df.iloc[-1]["valor"]
            ultima_fecha = df.iloc[-1]["fecha"].strftime("%d/%m/%Y")
            print(f"{nombre}: {ultimo_valor} ({ultima_fecha})")
        else:
            print(f"{nombre}: sin datos")
    return resultados

print("Función lista")

Función lista


In [10]:
# Celda 5 — Descargar datos reales del BCRP

SERIES_DIARIAS = {
    "tipo_cambio_venta":  "PD04638PD",
    "tipo_cambio_compra": "PD04637PD",
}

SERIES_MENSUALES = {
    "inflacion_12meses":  "PN01273PM",
    "tasa_interbancaria": "PN07819NM",
}

datos_diarios = fetch_todas_las_series(
    SERIES_DIARIAS,
    fecha_inicio, fecha_fin
)

datos_mensuales = fetch_todas_las_series(
    SERIES_MENSUALES,
    "2025-01-01", fecha_fin
)

tipo_cambio_venta: 3.439 (15/04/2026)
tipo_cambio_compra: 3.43585714285714 (15/04/2026)
inflacion_12meses: 3.80454581291585 (01/03/2026)
tasa_interbancaria: 4.2473 (01/03/2026)


In [11]:
### Celda 6 - Tabla resumen de indicadores

resumen = []

ultimo_venta = datos_diarios["tipo_cambio_venta"].iloc[-1]
penultimo_venta = datos_diarios["tipo_cambio_venta"].iloc[-2]
hace7_venta = datos_diarios["tipo_cambio_venta"].iloc[-7]

resumen.append({
    "indicador": "Tipo de cambio venta",
    "valor_actual": round(ultimo_venta["valor"], 4),
    "fecha": ultimo_venta["fecha"].strftime("%d/%m/%Y"),
    "var_1d": round(ultimo_venta["valor"] - penultimo_venta["valor"], 4),
    "var_7d": round(ultimo_venta["valor"] - hace7_venta["valor"], 4),
})

ultimo_inf = datos_mensuales["inflacion_12meses"].iloc[-1]
anterior_inf = datos_mensuales["inflacion_12meses"].iloc[-2]

resumen.append({
    "indicador": "Inflacion 12 meses",
    "valor_actual": round(ultimo_inf["valor"], 2),
    "fecha": ultimo_inf["fecha"].strftime("%d/%m/%Y"),
    "var_1d": None,
    "var_7d": round(ultimo_inf["valor"] - anterior_inf["valor"], 2),
})

ultimo_tasa = datos_mensuales["tasa_interbancaria"].iloc[-1]
anterior_tasa = datos_mensuales["tasa_interbancaria"].iloc[-2]

resumen.append({
    "indicador": "Tasa interbancaria",
    "valor_actual": round(ultimo_tasa["valor"], 2),
    "fecha": ultimo_tasa["fecha"].strftime("%d/%m/%Y"),
    "var_1d": None,
    "var_7d": round(ultimo_tasa["valor"] - anterior_tasa["valor"], 2),
})

df_resumen = pd.DataFrame(resumen)
df_resumen

,indicador,valor_actual,fecha,var_1d,var_7d
0,Tipo de cambio venta,3.439,15/04/2026,0.0471,0.0065
1,Inflacion 12 meses,3.800,01/03/2026,NaN,1.5900
2,Tasa interbancaria,4.250,01/03/2026,NaN,-0.0000


In [12]:
# Celda 7 — Perfil del usuario

perfil = {
    "ahorros":  "soles",
    "credito":  "si",
    "negocio":  "no",
}

print(perfil)

{'ahorros': 'soles', 'credito': 'si', 'negocio': 'no'}


In [13]:
# Celda 8 — Construir el contexto para OpenAI

tipo_cambio_hoy = df_resumen[df_resumen["indicador"] == "Tipo de cambio venta"]["valor_actual"].values[0]
tipo_cambio_var1d = df_resumen[df_resumen["indicador"] == "Tipo de cambio venta"]["var_1d"].values[0]
tipo_cambio_var7d = df_resumen[df_resumen["indicador"] == "Tipo de cambio venta"]["var_7d"].values[0]

inflacion_hoy = df_resumen[df_resumen["indicador"] == "Inflacion 12 meses"]["valor_actual"].values[0]

tasa_hoy = df_resumen[df_resumen["indicador"] == "Tasa interbancaria"]["valor_actual"].values[0]

contexto_economico = f"""
DATOS ECONÓMICOS DEL BCRP HOY:
- Tipo de cambio venta: S/ {tipo_cambio_hoy}
- Variación vs ayer: {tipo_cambio_var1d}
- Variación vs 7 días: {tipo_cambio_var7d}
- Inflación 12 meses: {inflacion_hoy}%
- Tasa interbancaria: {tasa_hoy}%

PERFIL DEL USUARIO:
- Ahorros en: {perfil["ahorros"]}
- Tiene crédito activo: {perfil["credito"]}
- Tiene negocio propio: {perfil["negocio"]}
"""

print(contexto_economico)


DATOS ECONÓMICOS DEL BCRP HOY:
- Tipo de cambio venta: S/ 3.439
- Variación vs ayer: 0.0471
- Variación vs 7 días: 0.0065
- Inflación 12 meses: 3.8%
- Tasa interbancaria: 4.25%

PERFIL DEL USUARIO:
- Ahorros en: soles
- Tiene crédito activo: si
- Tiene negocio propio: no



In [ ]:
# Celda 9 — Generar briefing con OpenAI

from openai import OpenAI

client = OpenAI()

system_prompt = """
Eres un asesor económico personal para ciudadanos peruanos.
Tu trabajo es explicar en lenguaje simple y directo cómo los 
indicadores económicos del día afectan la situación específica 
del usuario según su perfil.

Reglas:
- Usa lenguaje coloquial, como si hablaras con un amigo
- Sé específico — no des consejos genéricos
- Máximo 4 puntos concretos
- Cada punto debe terminar con una acción recomendada
- No uses jerga económica sin explicarla
- Siempre aclara que no eres asesor financiero certificado
"""

user_prompt = f"""
Basándote en estos datos y perfil, dime cómo me afecta 
la situación económica de hoy y qué debería considerar:

{contexto_economico}
"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt}
    ]
)

briefing = response.choices[0].message.content
print(briefing)